# SFNO: control vs. K&C bogus for Typhoon Mawar (WP02 2023)

Runs NVIDIA's SFNO (`sfno_73ch_small`, 0.25°, via earth2studio) twice from the
same ERA5 analysis at **2023-05-26 06Z**, about a day before Mawar's peak:

1. **control**: the unaltered ERA5 state;
2. **bogus**: the ERA5 state with a Kwon & Cheong (2009) vortex matched to the
   JTWC best-track intensity.

Both runs go 72 h in 6 h steps. At each step the storm is located by
`tcinit.rollout_tracking.track_rollout`, which centres each box on the previous
step's position, and the two runs are compared against the best track.

**Run on:** a cee-compute GPU node in `torch-venv` (see `envs/README.md`), with
`EARTH2STUDIO_CACHE` pointing at scratch. The initial state is pulled from the
ARCO ERA5 Zarr store on Google Cloud, so the node needs network access; the
first run downloads the SFNO checkpoint from NGC and the 73 ERA5 fields, both
cached under `EARTH2STUDIO_CACHE` afterwards.

**Resolution.** SFNO runs on the 0.25° ERA5 grid (about 28 km). Mawar's
best-track intensity at this time is about 165 kt with a radius of maximum
wind of a few tens of km, so the inner core of a K&C vortex matched to it
spans only a few grid cells. The grid cannot fully resolve the matched
vortex: expect the bogus initial MSLP minimum and peak wind on the grid to fall
short of the best-track values, and the model to spin part of it down early.

**Time axis.** SFNO takes a single state (no history), so there is no slot
ordering to check as in the Aurora notebook. The SFNO tensor is
`(batch, time, lead_time, variable, lat, lon)`; `time` is the initial time and
`lead_time` advances by 6 h per step. Cell 7 checks the lead time of every
state the iterator yields.

## 1. Configuration

In [ ]:
import os
from datetime import datetime, timedelta
from pathlib import Path

# Pick one GPU. Must be set before torch is imported.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

PROJECT = Path("/projects/aurora")
BEST_TRACK_FILE = PROJECT / "best-tracks/westpac/bwp2023/bwp022023.dat"
OUT_DIR = Path(f"/fgs/{os.environ['USER']}/causaltc/demo_out/sfno_mawar")

START_TIME = datetime(2023, 5, 26, 6)
STEP = timedelta(hours=6)
N_STEPS = 12  # 72 h
BOX_HALF_DEG = 10.0

if not BEST_TRACK_FILE.exists():
    raise FileNotFoundError(BEST_TRACK_FILE)
if "EARTH2STUDIO_CACHE" not in os.environ:
    raise RuntimeError("set EARTH2STUDIO_CACHE (see envs/README.md) before starting")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"output -> {OUT_DIR}")
print(f"earth2studio cache -> {os.environ['EARTH2STUDIO_CACHE']}")

In [ ]:
from collections import OrderedDict
from itertools import islice

import earth2studio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from earth2studio.data import ARCO_ERA5, fetch_data
from earth2studio.models.px import SFNO
from earth2studio.utils.time import to_time_array

from tcinit import BestTrack
from tcinit.backends import sfno_backend
from tcinit.constants import HPA_TO_PA
from tcinit.plotting import plot_snapshot
from tcinit.rollout_snapshots import plot_rollout_comparison, plot_rollout_snapshots
from tcinit.rollout_tracking import build_bogus, snapshot_from_box, track_rollout

DEVICE = torch.device("cuda")
print("torch", torch.__version__, "on", torch.cuda.get_device_name(0))
print("earth2studio", earth2studio.__version__)

## 2. Best track

In [ ]:
bt = BestTrack(BEST_TRACK_FILE)
bt_center = bt.get_location(START_TIME)
print(
    f"{bt.ID} at {START_TIME:%Y-%m-%d %HZ}: "
    f"({bt_center[0]:.1f}N, {bt_center[1]:.1f}E), "
    f"{bt.get_windspeed(START_TIME):.0f} m/s, "
    f"{bt.get_central_pressure(START_TIME) / HPA_TO_PA:.0f} hPa"
)

## 3. Load SFNO

The model is loaded before the initial state because its `input_coords()`
supply the variable list and lead time that `fetch_data` needs.
`load_default_package` points at the NGC `sfno_73ch_small` package and caches
it under `$EARTH2STUDIO_CACHE/sfno`.

In [ ]:
package = SFNO.load_default_package()
model = SFNO.load_model(package)
model = model.to(DEVICE)
model.eval()

in_coords = model.input_coords()
print("variables:", in_coords["variable"].size, " lead_time:", in_coords["lead_time"])
print(f"grid: {in_coords['lat'].size} x {in_coords['lon'].size} "
      f"(lat {in_coords['lat'][0]} .. {in_coords['lat'][-1]}, "
      f"lon {in_coords['lon'][0]} .. {in_coords['lon'][-1]})")

## 4. Initial state from ARCO ERA5

`fetch_data` returns a `(time, lead_time, variable, lat, lon)` tensor on the
GPU. A leading `batch` axis of size 1 is added so the tensor has the
six-dimensional layout `sfno_backend` expects. With a 6-D input,
earth2studio's batch wrapper keeps the `batch` axis on every state the
iterator yields; with the 5-D tensor it would strip it again.

The check at the end confirms that the grid is the one SFNO expects
(lat 90 → −90, lon 0 → 359.75) and that the analysis has a storm near the
best-track position.

In [ ]:
era5 = ARCO_ERA5()  # CHECK ON HPC: needs outbound access to the public ARCO GCS bucket (proxy_on on the Sandia network)
x5, c5 = fetch_data(
    source=era5,
    time=to_time_array([START_TIME]),
    variable=in_coords["variable"],
    lead_time=in_coords["lead_time"],
    device=DEVICE,
)
x_ctrl = x5.unsqueeze(0)
coords = OrderedDict([("batch", np.arange(1)), *c5.items()])
del x5

print("shape:", tuple(x_ctrl.shape), x_ctrl.dtype, x_ctrl.device)
print("dims: ", list(coords))
print("time: ", coords["time"], " lead_time:", coords["lead_time"])
assert list(coords) == ["batch", "time", "lead_time", "variable", "lat", "lon"]
assert np.array_equal(coords["variable"], in_coords["variable"])
assert np.array_equal(coords["lat"], in_coords["lat"])
assert np.array_equal(coords["lon"], in_coords["lon"])
assert coords["time"][0] == np.datetime64(START_TIME)

## 5. Control initial state

In [ ]:
def extract_box(state, center):
    # state is the (tensor, coords) pair yielded by the SFNO iterator.
    x, c = state
    return sfno_backend.extract_snapshot_dataset(
        x, c, box_center=center, box_size=BOX_HALF_DEG
    )


box_ctrl0 = extract_box((x_ctrl, coords), bt_center)
snap_ctrl0 = snapshot_from_box(box_ctrl0, valid_time=START_TIME, search_center=bt_center)
print(
    f"ERA5 storm at ({snap_ctrl0.loc[0]:.2f}, {snap_ctrl0.loc[1]:.2f}), "
    f"{snap_ctrl0.mslp_min / HPA_TO_PA:.1f} hPa | best track at "
    f"({bt_center[0]:.2f}, {bt_center[1]:.2f}), "
    f"{bt.get_central_pressure(START_TIME) / HPA_TO_PA:.0f} hPa"
)
plot_snapshot(
    snap_ctrl0,
    box_ctrl0,
    title=f"Mawar, ERA5 {START_TIME:%Y-%m-%d %HZ} (control initial state)",
)
plt.show()

## 6. Bogus initial state

`build_bogus` detects the storm, builds the K&C vortex from the best-track
intensity, and applies it to the box with `sfno_backend.apply`.
`sfno_backend.write_back` then writes the box in place into a copy of the
initial tensor; that copy is what the bogus run's iterator starts from.
`write_back` fills every `(batch, time, lead_time)` slot, and SFNO has only one
of each, so unlike Aurora there is no history slot to overwrite.

Modified: `msl, 2t, 10u, 10v`, `t, u, v, q, z` on the 13 levels, and surface
pressure `sp`, which the backend scales with the MSLP change
(`sp · msl_bogus / msl`) so the two stay consistent. `u100m, v100m, tcwv`
keep their ERA5 values in the storm core.

In [ ]:
x_bogus = x_ctrl.clone()
_, bv, box_bogus0 = build_bogus(
    box_ctrl0, bt, START_TIME, search_center=bt_center, apply=sfno_backend.apply
)
sfno_backend.write_back(
    x_bogus,
    coords,
    box_bogus0,
    box_lats=box_ctrl0.latitude.values,
    box_lons=box_ctrl0.longitude.values,
)

print(f"p_c  = {bv.p_c / HPA_TO_PA:.1f} hPa (best track)")
print(f"V_m  = {bv.V_m:.1f} m/s")
print(f"R_v  = {bv.R_v_km:.0f} km")

box_check = extract_box((x_bogus, coords), bt_center)
snap_bogus0 = snapshot_from_box(box_check, valid_time=START_TIME, search_center=bt_center)
print(
    f"MSLP min: control {snap_ctrl0.mslp_min / HPA_TO_PA:.1f} hPa -> "
    f"bogus {snap_bogus0.mslp_min / HPA_TO_PA:.1f} hPa on the 0.25 deg grid"
)
assert not torch.equal(x_bogus, x_ctrl), "write_back changed nothing"
plot_snapshot(
    snap_bogus0,
    box_check,
    title=f"Mawar, ERA5 + K&C bogus {START_TIME:%Y-%m-%d %HZ}",
)
plt.show()

## 7. Run and track both forecasts

`model.create_iterator(x, coords)` yields the initial condition first (lead
time 0), then one 6 h step per `next()`, without end. `sfno_states` takes the
first `N_STEPS + 1` of them, so `states[0]` is the state at `START_TIME` and
state `k` is valid at `START_TIME + k * STEP`; it asserts this from each
state's own coordinates. `track_rollout` consumes the states one at a time and
keeps only the storm box, so only the current global state is held on the GPU.

In [ ]:
def sfno_states(x, c):
    for k, (xs, cs) in enumerate(islice(model.create_iterator(x, c), N_STEPS + 1)):
        assert cs["lead_time"].size == 1 and cs["lead_time"][0] == np.timedelta64(6 * k, "h"), (
            k,
            cs["lead_time"],
        )
        assert cs["time"][0] + cs["lead_time"][0] == np.datetime64(START_TIME + k * STEP)
        yield xs, cs


common = dict(
    cyclone_id=bt.ID, start_time=START_TIME, initial_center=bt_center, step=STEP
)
with torch.inference_mode():
    ctrl = track_rollout(sfno_states(x_ctrl, coords), extract_box, **common)
    bogus = track_rollout(
        sfno_states(x_bogus, coords), extract_box, intervened=True, **common
    )

ctrl.to_netcdf(OUT_DIR / "control")
bogus.to_netcdf(OUT_DIR / "bogus")

## 8. Compare with the best track

In [ ]:
cols = {
    "model_lead_hours": "lead_h",
    "model_MSLP": "MSLP",
    "model_VMAX": "VMAX",
    "position_error_km": "pos_err_km",
}
df_ctrl = ctrl.to_comparison_dataframe(bt)
df_bogus = bogus.to_comparison_dataframe(bt)
table = pd.concat(
    {
        "control": df_ctrl[list(cols)].rename(columns=cols),
        "bogus": df_bogus[list(cols)].rename(columns=cols).drop(columns="lead_h"),
        "best track": df_ctrl[["bt_MSLP", "bt_VMAX"]].rename(
            columns={"bt_MSLP": "MSLP", "bt_VMAX": "VMAX"}
        ),
    },
    axis=1,
)
table.round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for df, label, style in ((df_ctrl, "control", "o-"), (df_bogus, "bogus", "^--")):
    lead = df["model_lead_hours"]
    axes[0].plot(lead, df["model_MSLP"], style, label=label)
    axes[1].plot(lead, df["model_VMAX"], style, label=label)
    axes[2].plot(lead, df["position_error_km"], style, label=label)
axes[0].plot(df_ctrl["model_lead_hours"], df_ctrl["bt_MSLP"], "k-", lw=2, label="best track")
axes[1].plot(df_ctrl["model_lead_hours"], df_ctrl["bt_VMAX"], "k-", lw=2, label="best track")
for ax, ylabel in zip(axes, ("min MSLP (hPa)", "max 10 m wind (kt)", "position error (km)")):
    ax.set_xlabel("lead time (h)")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle(f"SFNO, Mawar from {START_TIME:%Y-%m-%d %HZ}")
plt.tight_layout()
plt.show()

In [ ]:
plot_rollout_comparison(
    ctrl,
    bogus,
    OUT_DIR / "comparison_sfno_mawar.pdf",
    best_track=bt,
    labels=("SFNO control", "SFNO + K&C bogus"),
)
plot_rollout_snapshots(ctrl, OUT_DIR / "rollout_sfno_control.pdf", best_track=bt)
plot_rollout_snapshots(bogus, OUT_DIR / "rollout_sfno_bogus.pdf", best_track=bt)
print(sorted(p.name for p in OUT_DIR.glob("*.pdf")))